# Module 8: RLHF — Direct Preference Optimization (DPO)

> **Course: Training MiniMind LLM from Scratch**  |  *Google Colab NLP Course*

## 🎯 Learning Objectives
- Understand the journey from **RLHF → PPO → DPO**
- Derive and implement the DPO loss from first principles
- Train MiniMind to **prefer chosen over rejected** responses
- Compare DPO-aligned vs SFT baseline on sensitive prompts

---

## 🧠 From RLHF to DPO

### Classical RLHF (Reinforcement Learning from Human Feedback)
1. **Collect preferences**: human annotators label `(prompt, chosen, rejected)` triplets
2. **Train a reward model** $r_\phi(x, y)$ on the preferences
3. **RL fine-tuning**: use PPO to maximise $\mathbb{E}[r_\phi] - \beta \text{KL}(\pi_\theta \| \pi_\text{ref})$

### Problems with PPO-based RLHF
- Requires a **separate reward model** (another LLM to train and serve)
- PPO is **complex and unstable** (many hyperparameters)
- Very **memory intensive** (4 models in memory simultaneously)

### DPO: The Key Insight

DPO (Rafailov et al., 2023) shows that the optimal RLHF policy has a **closed form**:

$$\pi^*(y|x) \propto \pi_\text{ref}(y|x) \cdot \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

Solving for $r$ and plugging into the Bradley-Terry preference model yields the DPO loss — **no reward model needed!**

$$\mathcal{L}_\text{DPO} = -\mathbb{E}\!\left[\log \sigma\!\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_\text{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_\text{ref}(y_l|x)}\right)\right)\right]$$

Where:
- $y_w$ = **chosen** (preferred) response
- $y_l$ = **rejected** (dispreferred) response
- $\pi_\text{ref}$ = frozen reference model (our SFT checkpoint)
- $\beta$ = temperature controlling deviation from reference

**Intuition:** increase log-prob of chosen, decrease log-prob of rejected, *relative to the reference model*.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd): return subprocess.run(cmd, shell=True, check=False)

run("pip install -q torch transformers modelscope tqdm matplotlib")

if not os.path.exists('/content/minimind-colab'):
    run("git clone https://github.com/Boyu-Zhang-UOI/minimind-colab /content/minimind-colab")

sys.path.insert(0, '/content/minimind-colab')

run("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU'")

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 📥 Download DPO Dataset

In [ ]:
import os

dataset_dir = '/content/minimind-colab/dataset'
os.makedirs(dataset_dir, exist_ok=True)

dpo_file = f'{dataset_dir}/dpo.jsonl'
if not os.path.exists(dpo_file):
    print("Downloading dpo.jsonl …")
    !modelscope download --model gongjy/minimind_dataset dpo.jsonl --local_dir {dataset_dir}
else:
    print(f"✅ DPO dataset already present: {dpo_file}")

with open(dpo_file) as f:
    n_lines = sum(1 for _ in f)
print(f"DPO dataset: {n_lines:,} preference pairs")


## 📊 Visualising DPO Data

DPO data is structured as preference pairs: the **same prompt** with a *chosen* (preferred) and *rejected* (dispreferred) completion.

In [ ]:
import json

with open('/content/minimind-colab/dataset/dpo.jsonl') as f:
    samples = [json.loads(l) for l in f.readlines()[:5]]

print(f"Total samples inspected: {len(samples)}")
print(f"Keys in first sample: {list(samples[0].keys())}\n")

for i, s in enumerate(samples[:2]):
    print(f"{'='*60}")
    print(f"Sample {i+1}:")
    if 'chosen' in s:
        chosen_text   = s['chosen'] if isinstance(s['chosen'], str) else json.dumps(s['chosen'], ensure_ascii=False)
        rejected_text = s['rejected'] if isinstance(s['rejected'], str) else json.dumps(s['rejected'], ensure_ascii=False)
        print(f"  CHOSEN   : {chosen_text[:300]}")
        print(f"  REJECTED : {rejected_text[:300]}")
    else:
        print(json.dumps(s, ensure_ascii=False, indent=2)[:500])
    print()

# Quick stats
chosen_lens   = []
rejected_lens = []
for s in samples:
    if 'chosen' in s:
        chosen_lens.append(len(str(s['chosen'])))
        rejected_lens.append(len(str(s['rejected'])))
if chosen_lens:
    print(f"Avg chosen length   : {sum(chosen_lens)/len(chosen_lens):.0f} chars")
    print(f"Avg rejected length : {sum(rejected_lens)/len(rejected_lens):.0f} chars")


## ⚙️ DPO Configuration

In [ ]:
import argparse, os, torch

args = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='dpo',
    from_weight='full_sft',          # policy model starts from SFT checkpoint
    data_path='/content/minimind-colab/dataset/dpo.jsonl',
    hidden_size=768, num_hidden_layers=8, use_moe=0,
    epochs=1, batch_size=4, learning_rate=1e-5,
    beta=0.1,                        # DPO β — KL penalty weight
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16', num_workers=2,
    accumulation_steps=1, grad_clip=1.0,
    log_interval=50, save_interval=500,
    max_seq_len=512,
)
os.makedirs(args.save_dir, exist_ok=True)
print("DPO Configuration:")
for k, v in vars(args).items():
    print(f"  {k:<22} = {v}")


In [ ]:
import copy, sys, torch
sys.path.insert(0, '/content/minimind-colab')

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import DPODataset
from trainer.trainer_utils import setup_seed, get_model_params, SkipBatchSampler, get_lr, init_model
from contextlib import nullcontext
from torch import optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

setup_seed(42)
lm_config = MiniMindConfig(hidden_size=args.hidden_size,
                            num_hidden_layers=args.num_hidden_layers)

# ── Policy model (trainable) ──────────────────────────────────────────────
model, tokenizer = init_model(lm_config, args.from_weight,
                               tokenizer_path='/content/minimind-colab/model',
                               save_dir=args.save_dir, device=args.device)
print(f"✅ Policy model loaded from '{args.from_weight}' checkpoint")

# ── Reference model (frozen copy of SFT) ──────────────────────────────────
ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()
print(f"✅ Reference model: frozen copy (requires_grad=False for all {sum(p.numel() for p in ref_model.parameters())/1e6:.2f}M params)")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nPolicy model trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")


In [ ]:
import torch.nn.functional as F

def logits_to_log_probs(logits, labels):
    """Compute per-token log-probabilities for the label tokens."""
    # logits: (B, T, V), labels: (B, T)
    log_probs = F.log_softmax(logits, dim=-1)
    return torch.gather(log_probs, dim=2,
                        index=labels.unsqueeze(2)).squeeze(-1)


def dpo_loss(ref_log_probs, policy_log_probs, mask, beta=0.1):
    assert ref_log_probs.shape[0] % 2 == 0, "Batch size must be even (chosen + rejected pairs)"
    """
    DPO loss given concatenated chosen+rejected log-probs.

    The first half of the batch is 'chosen', the second half is 'rejected'.
    Mask marks response tokens (1 = contribute to loss, 0 = ignore).
    """
    # Sum log-probs over response tokens → scalar per sequence
    ref_lp    = (ref_log_probs    * mask).sum(dim=1)
    policy_lp = (policy_log_probs * mask).sum(dim=1)

    B    = ref_lp.shape[0]
    half = B // 2

    # Split into chosen and rejected
    chosen_ref  = ref_lp[:half];    rejected_ref  = ref_lp[half:]
    chosen_pol  = policy_lp[:half]; rejected_pol  = policy_lp[half:]

    # Log-ratio of policy over reference (implicit reward difference)
    pi_logratios  = chosen_pol  - rejected_pol   # should be > 0 after good training
    ref_logratios = chosen_ref  - rejected_ref

    # DPO objective: maximise margin between chosen and rejected
    loss = -F.logsigmoid(beta * (pi_logratios - ref_logratios))
    return loss.mean()


print("✅ logits_to_log_probs() and dpo_loss() defined")
print(f"   β = {args.beta}  (higher β = stronger KL penalty = stay closer to reference)")

# Quick sanity-check on toy data
B, T, V = 4, 16, 1000
toy_ref    = torch.randn(B, T, V)
toy_policy = toy_ref + 0.1 * torch.randn(B, T, V)  # policy slightly off reference
toy_labels = torch.randint(0, V, (B, T))
toy_mask   = torch.ones(B, T)
ref_lp     = logits_to_log_probs(toy_ref, toy_labels)
pol_lp     = logits_to_log_probs(toy_policy, toy_labels)
toy_loss   = dpo_loss(ref_lp, pol_lp, toy_mask, beta=args.beta)
print(f"   Sanity check loss (should be ~0.693 for random init): {toy_loss.item():.4f}")


## 🏋️ DPO Training Loop

Key differences from SFT:
1. Each batch contains **both chosen and rejected** sequences
2. We run **two forward passes** per batch: policy model (grad) + reference model (no grad)
3. The loss *compares* the two — no ground-truth label loss

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

loss_history, step_history = [], []

train_ds     = DPODataset(args.data_path, tokenizer, max_length=args.max_seq_len)
optimizer    = optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=0.01)
dtype_t      = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
autocast_ctx = (nullcontext() if 'cpu' in args.device
                else torch.cuda.amp.autocast(dtype=dtype_t))
scaler       = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

print(f"DPO dataset: {len(train_ds):,} preference pairs")

def dpo_train(epochs=1):
    model.train()
    global_step = 0
    for epoch in range(epochs):
        indices       = torch.randperm(len(train_ds)).tolist()
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader        = DataLoader(train_ds, batch_sampler=batch_sampler,
                                    num_workers=args.num_workers, pin_memory=True)
        iters = len(loader)
        pbar  = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, batch in enumerate(pbar, start=1):
            global_step += 1

            x_chosen   = batch['x_chosen'].to(args.device)
            x_rejected = batch['x_rejected'].to(args.device)
            y_chosen   = batch['y_chosen'].to(args.device)
            y_rejected = batch['y_rejected'].to(args.device)
            m_chosen   = batch['mask_chosen'].to(args.device)
            m_rejected = batch['mask_rejected'].to(args.device)

            # Concat chosen + rejected along batch dimension
            x    = torch.cat([x_chosen, x_rejected], dim=0)
            y    = torch.cat([y_chosen, y_rejected],  dim=0)
            mask = torch.cat([m_chosen, m_rejected],  dim=0)

            lr = get_lr(epoch * iters + step, epochs * iters, args.learning_rate)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            with autocast_ctx:
                # Reference forward (no grad)
                with torch.no_grad():
                    ref_out = ref_model(x)
                    ref_lp  = logits_to_log_probs(ref_out.logits[:, :-1], y[:, 1:])
                    ref_mask = mask[:, 1:]

                # Policy forward (with grad)
                pol_out = model(x)
                pol_lp  = logits_to_log_probs(pol_out.logits[:, :-1], y[:, 1:])

                loss = dpo_loss(ref_lp, pol_lp, ref_mask,
                                beta=args.beta) / args.accumulation_steps

            scaler.scale(loss).backward()

            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            cur = loss.item() * args.accumulation_steps
            loss_history.append(cur)
            step_history.append(global_step)
            pbar.set_postfix({'dpo_loss': f'{cur:.4f}', 'lr': f'{lr:.2e}'})

            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(12, 4))
                ax.plot(step_history, loss_history, 'b-', alpha=0.4,
                        linewidth=0.8, label='DPO Loss')
                if len(loss_history) > 20:
                    w  = 20
                    sm = [sum(loss_history[max(0,i-w):i+1]) / min(i+1, w)
                          for i in range(len(loss_history))]
                    ax.plot(step_history, sm, 'r-', linewidth=2, label='Smoothed')
                ax.axhline(y=0.693, color='gray', linestyle='--', alpha=0.5,
                           label='Random (ln2 ≈ 0.693)')
                ax.set_title(f'DPO Loss  (step {global_step})')
                ax.set_xlabel('Step'); ax.set_ylabel('Loss')
                ax.legend(); ax.grid(True, alpha=0.3)
                plt.tight_layout(); plt.show()

dpo_train(args.epochs)


In [ ]:
# Final loss plot after training completes
if loss_history:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(step_history, loss_history, 'b-', alpha=0.4, linewidth=0.8, label='DPO Loss')
    if len(loss_history) > 20:
        w  = 20
        sm = [sum(loss_history[max(0,i-w):i+1]) / min(i+1, w)
              for i in range(len(loss_history))]
        ax.plot(step_history, sm, 'r-', linewidth=2, label='Smoothed (w=20)')
    ax.axhline(y=0.693, color='gray', linestyle='--', alpha=0.5,
               label='Random baseline (ln 2)')
    ax.set_title('DPO Training Loss (full run)')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"Final loss: {loss_history[-1]:.4f}  (start: {loss_history[0]:.4f})")


## ⚖️ DPO vs SFT Comparison

DPO should improve on **alignment-sensitive** prompts — requests that an SFT model might answer in harmful or unhelpful ways.

Try prompts that test the model's ability to:
- Politely refuse inappropriate requests
- Give balanced answers on controversial topics
- Provide safe medical/legal/safety disclaimers

In [ ]:
import torch

def generate(m, tok, prompt, device='cuda', max_new_tokens=150, temperature=0.7):
    m.eval()
    conv  = [{"role": "user", "content": prompt}]
    text  = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    inp   = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(inp["input_ids"], attention_mask=inp["attention_mask"],
                         max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, pad_token_id=tok.pad_token_id,
                         eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

# Load SFT baseline for comparison
from model.model_minimind import MiniMindForCausalLM, MiniMindConfig
sft_model = MiniMindForCausalLM(lm_config)
sft_ckp   = f'{args.save_dir}/full_sft_{lm_config.hidden_size}.pth'
sft_model.load_state_dict(torch.load(sft_ckp, map_location=args.device), strict=False)
sft_model = sft_model.half().eval().to(args.device)

# Alignment-sensitive test prompts
test_prompts = [
    "如何提高工作效率？",
    "介绍几种放松减压的方法",
    "解释公平和平等的区别",
]

for p in test_prompts:
    print(f"\n{'='*60}")
    print(f"❓ {p}")
    sft_resp = generate(sft_model, tokenizer, p, args.device)
    dpo_resp = generate(model,     tokenizer, p, args.device)
    print(f"🟡 SFT : {sft_resp[:250]}")
    print(f"🟢 DPO : {dpo_resp[:250]}")

del sft_model


In [ ]:
import gc, torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


## 📝 Student Exercise

1. **Tune β**: Try `beta=0.01`, `0.1`, `0.5`, `1.0`.  
   Low β allows large deviations from the reference; high β keeps the policy conservative.  
   How does β affect both (a) the training loss curve and (b) response quality?

2. **Create preference pairs**: Write 5 custom `(prompt, chosen, rejected)` triplets in the DPO JSONL format.  
   Use them to make MiniMind prefer more concise answers. Verify with generation.

3. **Reward gap analysis**: After training, compute  
   $r_	heta(x, y_w) - r_	heta(x, y_l) = eta \log rac{\pi_	heta(y_w|x)}{\pi_	ext{ref}(y_w|x)} - eta \log rac{\pi_	heta(y_l|x)}{\pi_	ext{ref}(y_l|x)}$  
   on the test set. Plot the distribution. A positive mean gap indicates successful alignment.

## 💡 Discussion Questions

- DPO requires a **frozen reference model**. What happens if the reference is weak (e.g., pretrained rather than SFT)?
- Unlike PPO, DPO doesn't require **online rollouts**. What is lost by not sampling from the policy during training?
- **IPO** (Identity Preference Optimisation) replaces the log-sigmoid with an L2 penalty on the log-ratio.  
  When would IPO be preferable to DPO?
- How does DPO relate to **SPIN** (Self-Play Fine-Tuning)? What role does the reference model play in both?
